# ZS601 LiDAR A/B v3 — 固定 val10 诊断与独立几何开关

基于主基线开发，保留全部原数据。先选择 GPU 运行时，再依次运行。无需购买新额度。
`EXPERIMENT="A"/"B"` 控制组合；`OVERRIDES={"normal_loss":"off"}` 覆盖单项。
每1000步输出固定10相机的RGB、相机系法向、期望中心z-depth以及原始NPZ。
深度仅作诊断，不是LiDAR伪GT或无偏表面深度。代码的CPU测试通过；云端构建和训练需要本notebook实际验证。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys, subprocess, uuid, json, os, zipfile, shutil
REPO_URL = "https://github.com/VISjudy/ZS601_3DGS.git"
CODE_REF = '545bfc5f82f49fcce397f5bf45372e459ae7a30c'  # CSV + 1-sigma color ellipsoids
DATA_ZIP = Path("/content/drive/MyDrive/LCCDataset/ZS601meetingroom/ZS601meetingroom_data.zip")
RESULTS = Path("/content/drive/MyDrive/LCCDataset/zs601_output/gaussian-splattingWithMask_v3_cff221ccfb")
EXPERIMENT = "A"  # 改成 B 即开启五项约束
OVERRIDES = {}  # 例如 {"normal_loss": "off", "size_loss": "on"}
ITERATIONS = 30000
UNITS = "scene"  # 核实坐标单位后可改成 meters；不会缩放数据
# 固定val文件可以位于Drive其他目录。为空时只在解压数据内寻找唯一匹配。
FIXED_VAL_FILE = "/content/drive/MyDrive/LCCDataset/zs601_output/gaussian-splattingWithMask_v3_cff221ccfb/images-val10.txt"
TEST_FILE = "/content/drive/MyDrive/LCCDataset/zs601_output/gaussian-splattingWithMask_v3_cff221ccfb/images_test.txt"
POINT_FILE = ""  # 若有多个LAS，请显式填写选中的完整路径；不猜测3cm/4cm
IMAGES_POSES_FILE = ""
CAMERAS_FILE = ""
WORK = Path('/content') / ('zs601_v3_' + uuid.uuid4().hex[:10])
WORK.mkdir(exist_ok=False)
RESULTS.mkdir(parents=True, exist_ok=True)
def run(args, cwd=None):
    print(' '.join(map(str,args)), flush=True)
    args=list(map(str,args))
    logfile=WORK/('command_'+uuid.uuid4().hex[:10]+'.log')
    print('Log:',logfile,flush=True)
    with logfile.open('x') as log, subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True)
            log.write(line); log.flush()
        if proc.wait(): raise subprocess.CalledProcessError(proc.returncode,args)
run(['git','clone','--branch','v3-ab',REPO_URL,WORK/'repo'])
run(['git','checkout','--detach',CODE_REF],cwd=WORK/'repo')
CODE = WORK/'repo'/'gaussian-splattingWithMask_v3'
print('Pinned code:', CODE_REF, 'Workspace:', WORK)


In [ ]:
import torch
run(['nvidia-smi'])
run(['nvcc','--version'])
assert torch.cuda.is_available(), "请在运行时设置中选择GPU"
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,
       'gpu':torch.cuda.get_device_name(0)})
assert 'L4' in torch.cuda.get_device_name(0), '本轮要求L4，请修改运行时GPU'
# 保留Colab自带torch，编译主基线的扩展；不安装v2-dev扩展。
run([sys.executable,'-m','pip','install','ninja','plyfile','laspy','scipy','pillow',
     'opencv-python-headless','tqdm'])
os.environ['MAX_JOBS']='2'
run([sys.executable,'-m','pip','install','-v','--no-build-isolation',CODE/'submodules'/'simple-knn'])
run([sys.executable,'-m','pip','install','-v','--no-build-isolation',CODE/'submodules'/'diff-gaussian-rasterization'])
run([sys.executable,'-c','import torch; from simple_knn._C import distCUDA2; from diff_gaussian_rasterization import GaussianRasterizer; print("CUDA extensions import OK")'],cwd=CODE)
run([sys.executable,'-m','unittest','-v','test_geometry_v3'],cwd=CODE)


In [ ]:
# 只解压到新目录，不删除、不覆盖以前的数据。
assert DATA_ZIP.is_file(), f"找不到数据包：{DATA_ZIP}"
LOCAL_ZIP=WORK/'data.zip'
assert not LOCAL_ZIP.exists(), '请勿重复复制；数据准备成功后只重跑下方路径选择单元'
print('Copy Drive -> local:',DATA_ZIP,'->',LOCAL_ZIP,flush=True)
shutil.copy2(DATA_ZIP, LOCAL_ZIP)
print('Copied bytes:',LOCAL_ZIP.stat().st_size,flush=True)
EXTRACT=WORK/'extracted'; EXTRACT.mkdir()
with zipfile.ZipFile(LOCAL_ZIP) as z:
    for item in z.infolist():
        target=(EXTRACT/item.filename).resolve()
        if not target.is_relative_to(EXTRACT.resolve()):
            raise ValueError('ZIP路径越界：'+item.filename)
        if (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('ZIP包含符号链接，请检查：'+item.filename)
    z.extractall(EXTRACT)
print('Extracted locally:',EXTRACT,flush=True)


In [ ]:
# 缺少val文件时，填写FIXED_VAL_FILE后只重跑本单元，无需重新复制数据。
def local_input(path):
    p=Path(path).resolve()
    assert p.is_file(), str(p)
    if p.is_relative_to(WORK.resolve()): return p
    dest=WORK/('input_'+uuid.uuid4().hex[:10]+p.suffix)
    with p.open('rb') as src, dest.open('xb') as dst: shutil.copyfileobj(src,dst)
    print('Input copied locally:',p,'->',dest)
    return dest
def choose(explicit, pattern):
    if explicit:
        p=Path(explicit)
        assert p.is_file(), str(p)
        return local_input(p)
    matches=sorted(EXTRACT.rglob(pattern))
    if len(matches)!=1:
        raise ValueError(f"需要明确指定 {pattern}，候选：{matches}。固定val10缺失时请在配置里填已有文件的Drive路径；不会重新抽样。")
    return matches[0]
VAL=choose(FIXED_VAL_FILE,'images-val10.txt')
POSES=choose(IMAGES_POSES_FILE,'images.txt')
INTR=choose(CAMERAS_FILE,'cameras.txt')
POINTS=choose(POINT_FILE,'*.las')
tests=sorted(EXTRACT.rglob('images_test.txt'))
TEST=local_input(TEST_FILE) if TEST_FILE else (tests[0] if len(tests)==1 else None)
if len(tests)>1 and not TEST_FILE: raise ValueError(f'多个test文件，请指定：{tests}')
image_roots=[p.parent for p in EXTRACT.rglob('images') if p.is_dir() and (p.parent/'masks').is_dir()]
if len(image_roots)!=1: raise ValueError(f'无法唯一定位images/masks根目录：{image_roots}')
DATA=image_roots[0]
TRAIN=WORK/('images_train_v3_'+uuid.uuid4().hex[:10]+'.txt')
cmd=[sys.executable,'prepare_v3.py','--images_file',POSES,'--val_file',VAL,'--output_train',TRAIN]
if TEST: cmd+=['--test_file',TEST]
run(cmd,cwd=CODE)
print({'data':str(DATA),'points':str(POINTS),'train':str(TRAIN),'val':str(VAL),'test':str(TEST)})


若压缩包没有你固定的 `images-val10.txt`，上一个单元会明确停止；把原有文件的Drive路径填入配置，不会偷偷替换这10个视角。若没有test文件，仍排除val，但该次运行只能称固定val监控，不能称完整135张测试集实验。


In [ ]:
def train_command(experiment, output, iterations, overrides=None, resume=None):
    cmd=[sys.executable,'train_mask_v3.py','--experiment',experiment,
         '-s',DATA,'-m',output,'--point_cloud',POINTS,'--train_file',TRAIN,
         '--val_file',VAL,'--cameras_file',INTR,'--units',UNITS,
         '--iterations',iterations,'--position_lr_max_steps',iterations,
         '--val_interval',1000,'--checkpoint_interval',1000]
    if TEST: cmd+=['--test_file',TEST]
    for key,value in (overrides or {}).items(): cmd+=['--'+key,str(value)]
    if resume: cmd+=['--resume',resume]
    return cmd
# 真正的短冒烟：200步，使用B以覆盖五个几何模块；输出0/200步的全部val图。
SMOKE_OUT=RESULTS/('smoke_B_'+uuid.uuid4().hex[:10])
run(train_command('B',SMOKE_OUT,200),cwd=CODE)
print('Smoke output:',SMOKE_OUT)


In [ ]:
from IPython.display import display
from PIL import Image
for kind in ['rgb','normal','depth']:
    display(Image.open(SMOKE_OUT/'val_v3'/'iteration_000200'/f'val00_{kind}.png'))
print(json.loads((SMOKE_OUT/'val_v3'/'iteration_000200'/'manifest.json').read_text()))


确认冒烟数值和图像合理后，将下面 `RUN_FORMAL` 设为 True。每组使用独立目录；分别运行A/B时保持其他参数一致。这里不会自动启动两轮长训练。


In [ ]:
RUN_FORMAL=False
if RUN_FORMAL:
    OUTPUT=RESULTS/(EXPERIMENT+'_'+uuid.uuid4().hex[:10])
    run(train_command(EXPERIMENT,OUTPUT,ITERATIONS,OVERRIDES),cwd=CODE)
    print('Result:',OUTPUT)
else:
    print('正式训练尚未启动；请先检查200步冒烟结果。')


断线恢复：必须使用相同CODE_REF、数据、EXPERIMENT、OVERRIDES和ITERATIONS，从已完成checkpoint恢复到新目录。模型PLY不能代替checkpoint。首次恢复应检查恢复迭代、点数、loss及val图是否合理。


In [ ]:
RESUME_CHECKPOINT=""  # 例如 /content/drive/MyDrive/.../checkpoints/iteration_1000.pth
if RESUME_CHECKPOINT:
    RESUME_OUT=RESULTS/(EXPERIMENT+'_resume_'+uuid.uuid4().hex[:10])
    run(train_command(EXPERIMENT,RESUME_OUT,ITERATIONS,OVERRIDES,RESUME_CHECKPOINT),cwd=CODE)
